## This notebook provides functionality for data attribution for OpTC dataset and visualizing the top k attributions

In [ ]:
'''
Importing the require libraries here
'''
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch
from torch_geometric.data import Data
import os
import torch.nn.functional as F
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore')
from torch_geometric.loader import NeighborLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

In [ ]:
os.chdir("content")
%pwd

In [ ]:
from pprint import pprint
import gzip
from sklearn.manifold import TSNE
import json
import copy
import os

import gensim
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

import multiprocessing

In [ ]:
import math
class PositionalEncoding():

    def __init__(self, d_model ,max_len = 100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:,0::2] = torch.sin(position * div_term)
        self.pe[:,1::2] = torch.cos(position * div_term)

    def embed(self, x):
        x = x + self.pe[:x.size(0)]
        return x

encoder = PositionalEncoding(20)

In [ ]:
import gensim
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from collections import Counter
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

In [ ]:
def preprocess(data):
  new_data = {}
  for x in data:
    check1 = x['object'] in ['PROCESS','FILE','FLOW','MODULE']
    check2 = not (x['action'] in ['START','TERMINATE'])
    check3 = x['actorID'] != x['objectID']
    key = (x['action'],x['actorID'],x['objectID'],x['object'],x['pid'],x['ppid'])
    if check1 and check2 and check3:
      new_data[key] = x
  return list(new_data.values())

In [ ]:
def describe(x):
  action = x["action"]
  props = x['properties']
  typ = x['object']

  phrase = ''
  try:
    if typ == 'PROCESS':
        phrase = f"{props['parent_image_path']} {action} {props['image_path']} {props['command_line']}"    

    elif typ == 'FILE':
        phrase = f"{props['image_path']} {action} {props['file_path']}"    

    elif typ == 'FLOW':
        phrase = f"{props['image_path']} {action} {props['src_ip']} {props['src_port']} {props['dest_ip']} {props['dest_port']} {props['direction']}"    

    else:
        phrase = f"{props['image_path']} {action} {props['module_path']}"
  except:
      phrase = ''
  
  return phrase.split(' ')

In [ ]:
'''
The following function loads the data in streaming mode. It loads n number of rows from the dataset at a time.
It then creates a graphraph from these n rows and extract the features of the nodes in the graph.
We can define n using the batch parameter in the following function.
'''
def transform(text):
    lbldta = [get_labels(x) for x in text]
    lbldta = [x for x in lbldta if x != None]

    data = preprocess(lbldta)

    temp = [describe(x) for x in data]
    temp = [x for x in temp if len(x) != 0]

    for i in range(len(data)):
        data[i]['phrase'] = temp[i]

    df = pd.DataFrame.from_dict(data)
    df['timestamp'] = df['timestamp'].str[:-6]
    df['timestamp'] = pd.to_datetime(df['timestamp'],infer_datetime_format=True)
    df.sort_values(by='timestamp', ascending=True,inplace=True)

    return df

def get_labels(x):
  event = x
  typ = event['object']
  props = event['properties']
  try:
    if typ == "PROCESS":
      event["actorname"] = props['parent_image_path']
      event["objectname"] = props['image_path']

    if typ == "FILE":
      event["actorname"] = props['image_path'] 
      event["objectname"] = props['file_path']
    
    if typ == "MODULE":
      event["actorname"] = props['image_path']
      event["objectname"] = props['module_path']
    
    if typ == "FLOW":
      event["actorname"] = props['image_path']
      event["objectname"] = props['dest_ip']+' '+props['dest_port']
    
    return event
  except:
    return None

def load_data(name=None):
    if name == None:
        rawdata = []
        for i in range(1,2):
            f = open(f"singleHost/501_{i}.txt")
            content = [json.loads(line) for line in f]
            rawdata = rawdata + content
        return prepare_graph(transform(rawdata))
    else:
        f = open(name)
        content = [json.loads(line) for line in f]
        return prepare_graph(transform(content))

In [ ]:
import time
def prepare_graph(df):
  nodes = {}
  labels = {}
  edges = []
    
  dummies = {'PROCESS':0,'FLOW':1,'FILE':2,'MODULE':3}

  lblmap = {}
  neimap = {}

  for i in range(len(df)):
    x = df.iloc[i]

    actorid = x['actorID']
    objectid = x["objectID"]

    if not (actorid in nodes):
      nodes[actorid] = []
    if not (objectid in nodes):
      nodes[objectid] = []
    
    nodes[actorid] += x['phrase']
    nodes[objectid] += x['phrase']

    labels[actorid] = dummies['PROCESS']
    labels[objectid] = dummies[x['object']]
    
    lblmap[actorid] = x['actorname']
    lblmap[objectid] = x['objectname']
    
    if not (actorid in neimap):
      neimap[actorid] = set()
    if not (objectid in neimap):
      neimap[objectid] = set()
    
    neimap[actorid].add(x['objectname'])
    neimap[objectid].add(x['actorname'])
    
    if x['object'] == 'FLOW':
        edges.append(( actorid, objectid, x['properties']['direction'] ))
    else:
        edges.append(( actorid, objectid, x['action'] ))

  features = []
  feat_labels = []
  edge_index = [[],[]]
  index  = {}
  mapp = []
              
  for k,v in nodes.items():
    if not ((len(v)==1) and (v[0] == 'DELETE')):
        features.append(v)
        feat_labels.append(labels[k])
        index[k] = len(features) - 1
        mapp.append(k)

  for x in edges:
    src = index[x[0]]
    dst = index[x[1]]
    
    edge_index[0].append(src)
    edge_index[1].append(dst)    
    
  return features,feat_labels,edge_index,mapp,lblmap, neimap

In [ ]:
from torch_geometric.nn import GCNConv
from torch_geometric.nn import SAGEConv, GATConv
import torch.nn.functional as F
import torch.nn as nn

'''
Defining the model. The model consists mainly of graph sage and graph attention layers
'''
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SAGEConv(20, 32, normalize=True)
        self.conv2 = SAGEConv(32, 4, normalize=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return F.softmax(x, dim=1)

In [ ]:
from gensim.models.callbacks import CallbackAny2Vec

class EpochSaver(CallbackAny2Vec):
    '''Callback to save model after each epoch.'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        model.save('word2vec_optc.model')
        self.epoch += 1

In [ ]:
class EpochLogger(CallbackAny2Vec):
    '''Callback to log information about training'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        print("Epoch #{} start".format(self.epoch))

    def on_epoch_end(self, model):
        print("Epoch #{} end".format(self.epoch))
        self.epoch += 1

In [ ]:
logger = EpochLogger()
saver = EpochSaver()

word2vec = Word2Vec.load("word2vec2.model")

def infer(doc):  
  word_emb = []
  for word in doc:
    if word in word2vec.wv:
      word_emb.append(word2vec.wv[word])
  
  if len(word_emb) == 0:
    return np.zeros(30)

  out_emb = torch.tensor(word_emb,dtype=torch.float)
  if len(doc) < 100000:
    out_emb = encoder.embed(out_emb)
  out_emb = out_emb.detach().cpu().numpy()
  out_emb = np.mean(out_emb,axis=0)
  return out_emb

In [ ]:
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

In [ ]:
from itertools import compress
from torch_geometric import utils

def construct_neighborhood(ids,mapp,edges,hops):
    if hops == 0:
        return set()
    else:
        neighbors = set()
        for i in range(len(edges[0])):
            if mapp[edges[0][i]] in ids:
                neighbors.add(mapp[edges[1][i]])
            if mapp[edges[1][i]] in ids:
                neighbors.add(mapp[edges[0][i]])
        return neighbors.union( construct_neighborhood(neighbors,mapp,edges,hops-1) )

In [ ]:
def helper(MP,all_pids,GP,edges,mapp):
    
    TP = MP.intersection(GP)  
    FP = MP - GP              
    FN = GP - MP              
    TN = all_pids - (GP | MP)
    
    two_hop_gp = construct_neighborhood(GP,mapp,edges,0)
    two_hop_tp = construct_neighborhood(TP,mapp,edges,0)
    FPL = FP - two_hop_gp
    TPL = TP.union(FN.intersection(two_hop_tp))
    FN = FN - two_hop_tp

    TP,FP,FN,TN = len(TPL),len(FPL),len(FN),len(TN)
    
    FPR = FP / (FP+TN)
    TPR = TP / (TP+FN)

    print(f"Number of True Positives: {TP}")
    print(f"Number of Fasle Positives: {FP}")
    print(f"Number of False Negatives: {FN}")
    print(f"Number of True Negatives: {TN}\n")

    prec = TP / (TP + FP)
    print(f"Precision: {round(prec, 2)}")

    rec = TP / (TP + FN)
    print(f"Recall: {round(rec, 2)}")

    fscore = (2*prec*rec) / (prec + rec)
    print(f"Fscore: {round(fscore, 2)}\n")
    
    return TPL,FPL

In [ ]:
f = open(f"singleHost/501_1.txt")
content = [json.loads(line) for line in f]
df_ben = transform(content)
candidates = list(set(df_ben['actorname']))

In [ ]:
import os
import json
import pickle

import torch
import numpy as np
from torch.nn import CrossEntropyLoss
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric import utils
from sklearn.utils import class_weight

start = time.time()

all_score_matrices = {}
target = 'zeroth'

while len(candidates) > 0:
    print("Target:", target)
    
    #---------------------------------------------------------------------------
    #                           TRAINING SECTION
    #---------------------------------------------------------------------------

    model = GCN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    
    # Filter benign data to exclude the current target
    df_ben_trim = df_ben[df_ben["actorname"] != target]

    # Prepare the graph
    phrases, labels, edges, mapp, lbl, nemap = prepare_graph(df_ben_trim)
    
    # Infer embeddings for each node
    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)

    # Define our training loss
    criterion = CrossEntropyLoss()

    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float).to(device),
        y=torch.tensor(labels, dtype=torch.long).to(device),
        edge_index=torch.tensor(edges, dtype=torch.long).to(device)
    )

    for epochs in range(100):
        model.train()
        optimizer.zero_grad() 
        out = model(graph.x, graph.edge_index) 
        loss = criterion(out, graph.y) 
        loss.backward() 
        optimizer.step()      
        total_loss = loss.item() 
        #print(total_loss)

    torch.save(model.state_dict(), f"lword2vec_gnn_optc_explain.pth")

    #---------------------------------------------------------------------------
    #                           EVALUATION SECTION
    #---------------------------------------------------------------------------
    mode = "evaluation"
    
    # --- CACHING CHANGES ---
    eval_cache_file = "evaluation_cache_501.pkl"
    if os.path.exists(eval_cache_file):
        # If cache file already exists, load from it
        with open(eval_cache_file, "rb") as f:
            eval_cache = pickle.load(f)

        df = eval_cache["df"]
        all_ids = eval_cache["all_ids"]
        phrases = eval_cache["phrases"]
        labels = eval_cache["labels"]
        edges = eval_cache["edges"]
        mapp = eval_cache["mapp"]
        lbl = eval_cache["lbl"]
        nemap = eval_cache["nemap"]
        nodes = eval_cache["nodes"]
        gt_nodes = eval_cache["gt_nodes"]

        print("Loaded evaluation data from cache.")
    else:
        hosts = ["501"]
        all_events = []
        for h in hosts:
            path = f"../script_test/all_system_events"
            with open(path, "r") as f:
                raw_events = [json.loads(x) for x in f]
                all_events.extend(raw_events)
        
        df = transform(all_events)
        all_ids = set(list(df["actorID"]) + list(df["objectID"]))

        phrases, labels, edges, mapp, lbl, nemap = prepare_graph(df)
        nodes = [infer(x) for x in phrases]
        nodes = np.array(nodes)

        #with open("optc_groundtruth.txt") as f:
        #    tmp_nodes = f.read().split()
        gt_nodes = set(list(df["actorID"]))

        # Save data to cache
        eval_cache = {
            "df": df,
            "all_ids": all_ids,
            "phrases": phrases,
            "labels": labels,
            "edges": edges,
            "mapp": mapp,
            "lbl": lbl,
            "nemap": nemap,
            "nodes": nodes,
            "gt_nodes": gt_nodes,
        }
        with open(eval_cache_file, "wb") as f:
            pickle.dump(eval_cache, f)
        print("Evaluation data cached to disk.")

    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float).to(device),
        y=torch.tensor(labels, dtype=torch.long).to(device),
        edge_index=torch.tensor(edges, dtype=torch.long).to(device),
    )

    flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool)

    model.load_state_dict(
        torch.load(
            f"lword2vec_gnn_optc_explain.pth",
            map_location=torch.device(device)
        )
    )

    model.eval()
    out = model(graph.x, graph.edge_index)
    
    sorted, indices = out.sort(dim=1, descending=True)
    conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
    conf = (conf - conf.min()) / conf.max()
    scores = conf

    # Build a dictionary of scores for ground-truth nodes
    score_matrix = {}
    for x in gt_nodes:
        ind = mapp.index(x)
        # For demonstration, store the score under x's label
        score_matrix[lbl[x]] = float(scores[ind].item())

    # Store scores for this target in the global dictionary
    all_score_matrices[target] = score_matrix

    # Save all_score_matrices at the end of this iteration
    with open("all_score_matrices_501.pkl", "wb") as f:
        pickle.dump(all_score_matrices, f)

    # Move to the next target
    target = candidates.pop()

print("Done with all targets. Final all_score_matrices saved.")
end = time.time()

In [ ]:
import pickle

with open("all_score_matrices_501.pkl", "rb") as f:
    all_score_matrices = pickle.load(f)

def find_top_k_contributing_samples(all_score_matrices, k=3):
    """
    For each malicious process, find the top k omitted training processes 
    that cause the largest *increase* in anomaly score relative to baseline.

    Returns a dictionary of:
      malicious_process -> list of tuples (omitted_sample, score_difference),
    where the list is sorted by score_difference in descending order.
    """
    # Baseline scores (no training sample omitted)
    baseline_scores = all_score_matrices['zeroth']

    # This will store: malicious_process -> list of (omitted_sample, diff_from_baseline)
    top_k_contributions = {}

    # Loop through each malicious process in the baseline
    for malicious_proc, baseline_score in baseline_scores.items():
        # We'll accumulate (omitted_sample, diff) for all omitted samples
        diff_list = []

        # Compare scores from each omitted-sample scenario
        for omitted_sample, score_dict in all_score_matrices.items():
            # Skip the baseline key itself
            if omitted_sample == 'zeroth':
                continue

            # Get the anomaly score for the malicious process under this omission
            current_score = score_dict.get(malicious_proc)
            if current_score is None:
                # If for some reason this malicious_proc wasn't evaluated, skip
                continue

            # Calculate how much the anomaly score differs from the baseline
            diff_from_baseline = current_score - baseline_score

            diff_list.append((omitted_sample, diff_from_baseline))

        # Sort by difference in descending order (largest increase first)
        diff_list.sort(key=lambda x: x[1], reverse=True)

        # Take the top k
        top_k_contributions[malicious_proc] = diff_list[:k]

    return top_k_contributions

In [ ]:
import pickle

with open("all_score_matrices_501.pkl", "rb") as f:
    all_score_matrices = pickle.load(f)

In [ ]:
f = open(f"singleHost/501_1.txt")
content = [json.loads(line) for line in f]
df_ben = transform(content)

In [ ]:
f = open(f"../script_test/all_system_events")
content = [json.loads(line) for line in f]
df_mal = transform(content)

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

def get_graph(df, target_node, object_type=['PROCESS']):
    df_filtered = df[df['object'].isin(object_type)]
    
    df_filtered['actorname'] = df_filtered['actorname'].str.split('\\').str[-1]
    df_filtered['objectname'] = df_filtered['objectname'].str.split('\\').str[-1]

    file_extensions = ['.dll', '.DLL', '.cat', '.lst', '.jpg', '.psd1']
    edges = df_filtered[['actorname', 'objectname']].values.tolist()
    edges = [x for x in edges if not any(ext in x[1] for ext in file_extensions)]

    G = nx.Graph()
    G.add_edges_from(edges)

    self_loops = list(nx.selfloop_edges(G))
    G.remove_edges_from(self_loops)

    distances = nx.single_source_shortest_path_length(G, target_node, cutoff=2)
    two_hop_nodes = list(distances.keys())
    subgraph = G.subgraph(two_hop_nodes).copy()

    return subgraph

In [ ]:
import networkx as nx

def trim_graph_from_outside(G1: nx.Graph, G2: nx.Graph, target: str):
    changed = True
    while changed:
        changed = False
        
        distances = nx.single_source_shortest_path_length(G1, target)
        if len(distances) == 1:
            break
        
        max_dist = max(distances.values())
        
        outer_layer = [n for n, dist in distances.items() 
                       if dist == max_dist and n != target]
        
        if not outer_layer:
            break
        
        for n in outer_layer:
            if n not in G2:
                continue
            
            G1_neighbors = set(G1.neighbors(n))
            G2_neighbors = set(G2.neighbors(n))
            
            if G1_neighbors == G2_neighbors:
                G1.remove_node(n)
                changed = True
            else:
                overlap_1hop = G1_neighbors.intersection(G2_neighbors)
                
                for neighbor in overlap_1hop:
                    if G1.has_edge(n, neighbor):
                        G1.remove_edge(n, neighbor)
                        changed = True
    return G1

In [ ]:
benign_labels = []
target_node = 'powershell'
results = find_top_k_contributing_samples(all_score_matrices, k=3)
for malicious_proc, top_list in results.items():
    if target_node in malicious_proc:
        print(f"Malicious process: {malicious_proc}")
        for (omitted_sample, diff) in top_list:
            omitted_filename = omitted_sample#.split('\\')[-1]
            benign_labels.append(omitted_filename)
            print(f"  Omitted sample: {omitted_filename}, Score difference: {diff:.4f}")
        print()

In [ ]:
targets = list(results.keys())
targets = [x.split('\\')[-1] for x in targets]

for target_node in targets:
    try:
        benign_labels = []
        results = find_top_k_contributing_samples(all_score_matrices, k=3)
        for malicious_proc, top_list in results.items():
            if target_node in malicious_proc:
                for (omitted_sample, diff) in top_list:
                    omitted_filename = omitted_sample.split('\\')[-1]
                    benign_labels.append(omitted_filename)
    
        H1 = get_graph(df_mal, target_node)
        
        print("Original: ", target_node, len(H1.edges()))
        for bl in benign_labels:
            Hb = get_graph(df_ben, bl)
            H1 = trim_graph_from_outside(H1, Hb, target_node)
            print('Attribution: ', bl, len(Hb.edges()))
        print("Reduced: ", target_node, len(H1.edges()))
        print()
    except:
        pass

In [ ]:
#target_node = 'lsass.exe'
H1 = get_graph(df_mal, target_node)

In [ ]:
print(len(H1.edges))
for bl in benign_labels:
    Hb = get_graph(df_ben, bl)
    H1 = trim_graph_from_outside(H1, Hb, target_node)
    print(bl,len(H1.edges()))

In [ ]:
H = H1

net = Network(
    notebook=True,
    height="750px",
    width="100%",
    bgcolor="#222222",
    font_color="white",
    cdn_resources='in_line'
)

for node in H.nodes():
    if node == target_node:
        net.add_node(
            node,
            title=node,
            label=node,
            color="red"   
        )
    else:
        net.add_node(
            node,
            title=node,
            label=node,
            color="green" 
        )

for source, target in H.edges():
    if source == target_node or target == target_node:
        net.add_edge(source, target, color="red")
    else:
        net.add_edge(source, target, color="gray")

net.set_options("""
var options = {
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -26,
            "centralGravity": 0.005,
            "springLength": 230,
            "springConstant": 0.18
        },
        "maxVelocity": 146,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {"iterations": 150}
    },
    "nodes": {
        "font": {
            "size": 12
        }
    }
}
""")

net.show("../graph_explanation.html")

In [ ]:
H = H1

net = Network(
    notebook=True,
    height="750px",
    width="100%",
    bgcolor="#222222",
    font_color="white",
    cdn_resources='in_line'
)

for node in H.nodes():
    if node == target_node:
        net.add_node(
            node,
            title=node,
            label=node,
            color="red"   
        )
    else:
        net.add_node(
            node,
            title=node,
            label=node,
            color="green" 
        )

for source, target in H.edges():
    if source == target_node or target == target_node:
        net.add_edge(source, target, color="red")
    else:
        net.add_edge(source, target, color="gray")

net.set_options("""
var options = {
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -26,
            "centralGravity": 0.005,
            "springLength": 230,
            "springConstant": 0.18
        },
        "maxVelocity": 146,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {"iterations": 150}
    },
    "nodes": {
        "font": {
            "size": 12
        }
    }
}
""")

net.show("../graph_explanation.html")